# To Do:

- Create new entry method for retrieval with reranking (with appropriate top_p, top_k, top_n parameters and the parameters for hybrid / mmr / recency bias)

# Retrieval Agentic Workflow

In [4]:
from typing import TypedDict, List
from langchain_core.messages import AnyMessage
from typing import TypedDict, List, Annotated
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.graph.message import add_messages
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from langchain_core.messages import ToolMessage
from langgraph.prebuilt import create_react_agent
# --- State ---
from typing import TypedDict, List, Annotated
from langchain_core.messages import AnyMessage
import operator



# --- State ---
class AgentState(TypedDict):
    user_input: str
    # Use add_messages reducer for messages to handle concurrent appends
    messages: Annotated[List[AnyMessage], add_messages]
        
    # These are single values, so they're fine as-is
    has_context: bool
    final_answer: str
    # Add this to track the retrieval agent message
    retrieval_agent_message: AnyMessage

In [5]:
from langchain_core.tools import tool
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    max_output_tokens=1000  # 👈 Limit output to 256 token, be careful token limit affects tool calling ability
)

# Retrieval

In [27]:
import os
import re
from typing import List, Dict, Optional
from dotenv import load_dotenv
import psycopg2
from psycopg2.extras import RealDictCursor
from sentence_transformers import SentenceTransformer
from huggingface_hub import login
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import logging


load_dotenv()

login(token=os.getenv("HUGGINGFACE_TOKEN"))
CONNECTION_STRING = os.getenv("DATABASE_URL")
SECRET_KEY = os.getenv("DATABASE_ENCRYPTION_KEY")

engine = create_engine(
    CONNECTION_STRING,
    pool_size=5,
    max_overflow=10,
    pool_pre_ping=True,
    pool_recycle=3600,
    connect_args={
        "keepalives": 1,
        "keepalives_idle": 30,
        "keepalives_interval": 10,
        "tcp_user_timeout": 60000,
    },
    echo=False
)

# Load model once globally
bi_encoder = SentenceTransformer("google/embeddinggemma-300m")

In [ ]:
def embed(text: str) -> list:
    """Generate embedding vector from text"""
    if not text or not isinstance(text, str):
        raise ValueError("Query must be a non-empty string")
    return bi_encoder.encode(text, normalize_embeddings=True).tolist()  # ✅ Normalize for cosine similarity

# Retrieval Functions (HYBRID VECTOR + BM25)

- embedding search through vector similarity
- bm25 search through exact match and fuzzy match 
- result are merge through weighted function 
    - hybrid_score = α * bm25_score + (1 - α) * emb_score

**Args**:
- `query`: input string
- `top_k`: number of results for each retrieval, and then number of results to keep after hybrid ranking
- `threshold`: minimum similarity cutoff (for embeddings only)
- `distance`: minimum edit distance for fuzzy match BM25
- `alpha`: weighting factor:
    - α = 0 → embeddings only.
    - α = 1 → BM25 only.

**Techinical Details**:
- vector search:
    - the vector embedding is compared based on `content` column in STM, `description` in HCM and `value` in LTM -- the raw information content
    - the bm25 indexes are implemented on
- BM25 search:
    - `content` column in STM
    - `category_search`, `key`, `value` (with `elderly_id` as an exact-match filter)
    - `record_type_search`, `description` (with `elderly_id` as an exact-match filter)

    
**Notes**:
- because the BM25 uses the elderly_id as an index, the bm25 scores are inflated, though inflated equally across bm25 results.
- but the functions already implements a normalising in the bm25 score before hybrid ranking


## Functions

In [8]:
ELDERLY_ID = "12345678-1234-1234-1234-012345678910"

### STM

In [9]:
def retrieve_hybrid_stm(query: str, top_k: int = 5, threshold: float = 0.3,
                        distance: int = 2, alpha: float = 0.5):
    try:
        emb = embed(query)

        # --- Embeddings ---
        sql_emb = text(f"""
            WITH nearest AS MATERIALIZED (
                SELECT
                    id, content, created_at, embedding,
                    embedding <=> (:emb)::vector AS distance
                FROM short_term_memory
                WHERE elderly_id = :elderly_id
                ORDER BY distance
                LIMIT :top_k
            )
            SELECT id, content, created_at, embedding, 1 - distance AS similarity
            FROM nearest
            {"WHERE 1 - distance >= :threshold" if threshold is not None else ""}
            ORDER BY distance
            LIMIT :top_k;
        """)
        params_emb = {"emb": str(emb), "elderly_id": ELDERLY_ID, "top_k": top_k}
        if threshold is not None:
            params_emb["threshold"] = threshold

        with engine.connect() as conn:
            rows_emb = conn.execute(sql_emb, params_emb).fetchall()

        emb_results = {
            r.id: {
                "id": r.id,
                "content": r.content,
                "created_at": r.created_at,
                "embedding": r.embedding,  # ✅ Added
                "emb_score": float(r.similarity)
            }
            for r in rows_emb
        }

        # --- BM25 ---
        query = query.replace("'", "''")

        sql_bm25 = text("""
            SELECT id, content, created_at, embedding, paradedb.score(id) AS bm25_score
            FROM short_term_memory
            WHERE elderly_id = :elderly_id
              AND (content @@@ :query OR id @@@ paradedb.match('content', :query, distance => :distance))
            ORDER BY bm25_score DESC
            LIMIT :top_k;
        """)
        params_bm25 = {"elderly_id": ELDERLY_ID, "query": query, "distance": distance, "top_k": top_k}
        with engine.connect() as conn:
            rows_bm25 = conn.execute(sql_bm25, params_bm25).fetchall()

        max_bm25 = max((float(r.bm25_score) for r in rows_bm25), default=1.0)
        bm25_results = {
            r.id: {
                "id": r.id,
                "content": r.content,
                "created_at": r.created_at,
                "embedding": r.embedding,  # ✅ Added
                "bm25_score": float(r.bm25_score) / max_bm25
            }
            for r in rows_bm25
        }

        # --- Merge + hybrid ---
        combined = {}
        for id_, r in {**emb_results, **bm25_results}.items():
            emb_score = emb_results.get(id_, {}).get("emb_score", 0.0)
            bm25_score = bm25_results.get(id_, {}).get("bm25_score", 0.0)
            hybrid = alpha * bm25_score + (1 - alpha) * emb_score
            combined[id_] = {
                **r,
                "emb_score": emb_score,
                "bm25_score": bm25_score,
                "hybrid_score": round(hybrid, 4)
            }

        return sorted(combined.values(), key=lambda x: x["hybrid_score"], reverse=True)[:top_k]

    except Exception as e:
        logging.warning(f"❌ Failed hybrid STM retrieval: {str(e)}")
        return []

### LTM

In [10]:
def retrieve_hybrid_ltm(query: str, top_k: int = 5, threshold: float = 0.3,
                        distance: int = 2, alpha: float = 0.5):
    try:
        emb = embed(query)

        # --- Embeddings ---
        sql_emb = text(f"""
            WITH nearest AS MATERIALIZED (
                SELECT id, category, key, value, last_updated, embedding,
                       embedding <=> (:emb)::vector AS distance
                FROM long_term_memory
                WHERE elderly_id = :elderly_id
                ORDER BY distance
                LIMIT :top_k
            )
            SELECT id, category, key, value, last_updated, embedding, 1 - distance AS similarity
            FROM nearest
            {"WHERE 1 - distance >= :threshold" if threshold is not None else ""}
            ORDER BY distance
            LIMIT :top_k;
        """)
        params_emb = {"emb": str(emb), "elderly_id": ELDERLY_ID, "top_k": top_k}
        if threshold is not None:
            params_emb["threshold"] = threshold

        with engine.connect() as conn:
            rows_emb = conn.execute(sql_emb, params_emb).fetchall()

        emb_results = {
            r.id: {
                "id": r.id,
                "category": r.category,
                "key": r.key,
                "value": r.value,
                "last_updated": r.last_updated,
                "embedding": r.embedding,  # ✅ Added
                "emb_score": float(r.similarity)
            }
            for r in rows_emb
        }

        # --- BM25 ---
        query = query.replace("'", "''")

        sql_bm25 = text("""
            SELECT id, category, key, value, last_updated, embedding, paradedb.score(id) AS bm25_score
            FROM long_term_memory
            WHERE elderly_id = :elderly_id
              AND (
                category_search @@@ :query OR key @@@ :query OR value @@@ :query
                OR id @@@ paradedb.match('category_search', :query, distance => :distance)
                OR id @@@ paradedb.match('key', :query, distance => :distance)
                OR id @@@ paradedb.match('value', :query, distance => :distance)
              )
            ORDER BY bm25_score DESC
            LIMIT :top_k;
        """)
        params_bm25 = {
            "elderly_id": ELDERLY_ID,
            "query": query,
            "distance": distance,
            "top_k": top_k
        }

        with engine.connect() as conn:
            rows_bm25 = conn.execute(sql_bm25, params_bm25).fetchall()

        max_bm25 = max((float(r.bm25_score) for r in rows_bm25), default=1.0)
        bm25_results = {
            r.id: {
                "id": r.id,
                "category": r.category,
                "key": r.key,
                "value": r.value,
                "last_updated": r.last_updated,
                "embedding": r.embedding,  # ✅ Added
                "bm25_score": float(r.bm25_score) / max_bm25
            }
            for r in rows_bm25
        }

        # --- Merge + hybrid ---
        combined = {}
        all_ids = set(emb_results.keys()) | set(bm25_results.keys())

        for id_ in all_ids:
            emb_data = emb_results.get(id_, {
                "id": id_,
                "category": "",
                "key": "",
                "value": "",
                "last_updated": None,
                "embedding": [],  # ✅ Default empty list
                "emb_score": 0.0
            })
            bm25_data = bm25_results.get(id_, {
                "id": id_,
                "category": "",
                "key": "",
                "value": "",
                "last_updated": None,
                "embedding": [],  # ✅ Default empty list
                "bm25_score": 0.0
            })

            combined[id_] = {
                "id": id_,
                "category": emb_data["category"] or bm25_data["category"],
                "key": emb_data["key"] or bm25_data["key"],
                "value": emb_data["value"] or bm25_data["value"],
                "last_updated": emb_data["last_updated"] or bm25_data["last_updated"],
                "embedding": emb_data["embedding"] or bm25_data["embedding"],  # ✅ Merge embedding
                "emb_score": emb_data.get("emb_score", 0.0),
                "bm25_score": bm25_data.get("bm25_score", 0.0),
                "hybrid_score": round(
                    alpha * bm25_data.get("bm25_score", 0.0) +
                    (1 - alpha) * emb_data.get("emb_score", 0.0),
                    4
                )
            }

        return sorted(combined.values(), key=lambda x: x["hybrid_score"], reverse=True)[:top_k]

    except Exception as e:
        logging.warning(f"❌ Failed hybrid LTM retrieval: {str(e)}")
        return []

### HCM

In [11]:
def retrieve_hybrid_health(query: str, top_k: int = 5, threshold: float = 0.3,
                         distance: int = 2, alpha: float = 0.5):
    try:
        emb = embed(query)

        # --- Embeddings ---
        sql_emb = text(f"""
            WITH nearest AS MATERIALIZED (
                SELECT id, record_type, description, diagnosis_date, last_updated, embedding,
                       embedding <=> (:emb)::vector AS distance
                FROM healthcare_records
                WHERE elderly_id = :elderly_id
                ORDER BY distance
                LIMIT :top_k
            )
            SELECT id, record_type, description, diagnosis_date, last_updated, embedding, 1 - distance AS similarity
            FROM nearest
            {"WHERE 1 - distance >= :threshold" if threshold is not None else ""}
            ORDER BY distance
            LIMIT :top_k;
        """)
        params_emb = {"emb": str(emb), "elderly_id": ELDERLY_ID, "top_k": top_k}
        if threshold is not None:
            params_emb["threshold"] = threshold

        with engine.connect() as conn:
            rows_emb = conn.execute(sql_emb, params_emb).fetchall()

        emb_results = {
            r.id: {
                "id": r.id,
                "record_type": r.record_type,
                "description": r.description,
                "diagnosis_date": r.diagnosis_date.isoformat() if r.diagnosis_date else None,
                "last_updated": r.last_updated.isoformat() if r.last_updated else None,
                "embedding": r.embedding,  # ✅ Added
                "emb_score": float(r.similarity)
            }
            for r in rows_emb
        }

        # --- BM25 ---
        query = query.replace("'", "''")

        sql_bm25 = text("""
            SELECT id, record_type, description, diagnosis_date, last_updated, embedding, paradedb.score(id) AS bm25_score
            FROM healthcare_records
            WHERE elderly_id = :elderly_id
              AND (
                record_type_search @@@ :query OR description @@@ :query
                OR id @@@ paradedb.match('record_type_search', :query, distance => :distance)
                OR id @@@ paradedb.match('description', :query, distance => :distance)
              )
            ORDER BY bm25_score DESC
            LIMIT :top_k;
        """)
        params_bm25 = {
            "elderly_id": ELDERLY_ID,
            "query": query,
            "distance": distance,
            "top_k": top_k
        }

        with engine.connect() as conn:
            rows_bm25 = conn.execute(sql_bm25, params_bm25).fetchall()

        max_bm25 = max((float(r.bm25_score) for r in rows_bm25), default=1.0)
        bm25_results = {
            r.id: {
                "id": r.id,
                "record_type": r.record_type,
                "description": r.description,
                "diagnosis_date": r.diagnosis_date.isoformat() if r.diagnosis_date else None,
                "last_updated": r.last_updated.isoformat() if r.last_updated else None,
                "embedding": r.embedding,  # ✅ Added
                "bm25_score": float(r.bm25_score) / max_bm25
            }
            for r in rows_bm25
        }

        # --- Merge + hybrid ---
        combined = {}
        all_ids = set(emb_results.keys()) | set(bm25_results.keys())

        for id_ in all_ids:
            emb_data = emb_results.get(id_, {
                "id": id_,
                "record_type": "",
                "description": "",
                "diagnosis_date": None,
                "last_updated": None,
                "embedding": [],  # ✅ Default empty list
                "emb_score": 0.0
            })
            bm25_data = bm25_results.get(id_, {
                "id": id_,
                "record_type": "",
                "description": "",
                "diagnosis_date": None,
                "last_updated": None,
                "embedding": [],  # ✅ Default empty list
                "bm25_score": 0.0
            })

            combined[id_] = {
                "id": id_,
                "record_type": emb_data["record_type"] or bm25_data["record_type"],
                "description": emb_data["description"] or bm25_data["description"],
                "diagnosis_date": emb_data["diagnosis_date"] or bm25_data["diagnosis_date"],
                "last_updated": emb_data["last_updated"] or bm25_data["last_updated"],
                "embedding": emb_data["embedding"] or bm25_data["embedding"],  # ✅ Merge embedding
                "emb_score": emb_data.get("emb_score", 0.0),
                "bm25_score": bm25_data.get("bm25_score", 0.0),
                "hybrid_score": round(
                    alpha * bm25_data.get("bm25_score", 0.0) +
                    (1 - alpha) * emb_data.get("emb_score", 0.0),
                    4
                )
            }

        return sorted(combined.values(), key=lambda x: x["hybrid_score"], reverse=True)[:top_k]

    except Exception as e:
        logging.warning(f"❌ Failed hybrid health retrieval: {str(e)}")
        return []

### Testing the retrieval functions

In [16]:
retrieve_hybrid_ltm("jonathan is my son's name?")

[{'id': UUID('22fda1a6-ea5e-46b7-9f47-3dfb3708d2a9'),
  'category': 'family',
  'key': 'son_name',
  'value': 'Jonathan',
  'last_updated': datetime.datetime(2025, 9, 17, 12, 3, 20, 659230),
  'embedding': '[-0.21277629,-6.598885e-05,0.018334383,-0.015835546,0.015027604,0.02448936,-0.04352825,0.033948537,0.008696369,-0.056709293,-0.020841662,-0.046240456,0.03962277,-0.031976063,0.0905365,0.022883626,0.02401342,-0.044154745,-0.053684834,0.017803276,0.0035022057,-0.02730019,-0.015853776,-0.018999172,-0.015485527,0.036325656,0.030795233,0.008346978,-0.007626878,-0.039284118,0.05456175,0.0095214415,0.025097672,0.010246136,0.004878111,0.0573215,0.026146349,-0.07749279,0.0423038,-0.02950501,-0.06635376,0.051750742,-0.008201085,-0.017909564,0.010647753,-0.014542041,-0.0356775,-0.023334019,-0.0033042468,0.024194114,0.011199333,0.0065349704,-0.04756231,-0.003987571,-0.03576272,-0.031174403,-0.028583938,-0.01238028,-0.05176874,0.047213234,-0.025149252,-0.012174051,-0.011255095,-0.009305863,0.028

# Reranking

### MMR with cross-enconder reranking 
- Reranks results using Maximal Marginal Relevance (MMR) with cross-encoder relevance and cosine similarity for redundancy, incorporating recency score.

**Args**:
- `query` (str): Original user query.
- `results` (List[Dict]): Output from `retrieve_hybrid_stm()`
- `cross_encoder` (CrossEncoder): Pre-loaded model (e.g., BAAI/bge-reranker-base)
- `alpha` (float): MMR trade-off between relevance and diversity (0=diverse, 1=relevant)
- `recency_weight` (float): Weight to multiply recency_score into relevance score
- `top_k` (int): Final number of results to return
- `device` (str): Device for cross-encoder (ignored if model already loaded)

**Returns**:
- `List[Dict]`: Reranked results with added 'mmr_score' and 'relevance_score'

In [ ]:
from typing import List, Dict, Any
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder
from time_importance_score import compute_recency_score

cross_encoder = CrossEncoder('BAAI/bge-reranker-base', device='cpu', trust_remote_code=True)

In [124]:
import numpy as np
from typing import List, Dict, Any
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder


def rerank_with_mmr_and_recency(
    query: str,
    results: List[Dict[str, Any]],
    cross_encoder: CrossEncoder,
    alpha: float = 0.7,   # MMR balance: relevance vs diversity
    beta: float = 0.1,    # Small bonus for recency
    top_k: int = 5,
) -> List[Dict[str, Any]]:
    if not results:
        return []
    
    #################################################################
    # --- Extracting relevant metadata about information chunks --- #
    #################################################################

    # ensure recency scores exist (already scaled 0–1)
    results = compute_recency_score(results, query)

    # extract texts
    texts = []
    for r in results:
        text = r.get("content") or r.get("value") or r.get("description")
        if not isinstance(text, str) or not text.strip():
            raise ValueError("Each result must have one of 'content', 'value', or 'description' as non-empty string.")
        texts.append(text)

    # extract embeddings
    embeddings = []
    for r in results:
        emb_str = r.pop("embedding", None)
        emb_list = [float(x) for x in emb_str.strip("[]").split(",")]  # embeddings are saved as strings
        embeddings.append(emb_list)
    embeddings = np.array(embeddings, dtype=np.float32)

    # recency is already normalized [0,1]
    recency_normalized = np.array([r.get("recency_score", 0.0) for r in results], dtype=np.float32)

    #################################################################
    # ---               Computing CE Relevance                    --- #
    #################################################################
    
    # relevance from cross-encoder
    pairs = [[query, text] for text in texts]
    ce_raw_scores = cross_encoder.predict(pairs)

    # normalize cross_encoder scores [0,1]
    min_score, max_score = ce_raw_scores.min(), ce_raw_scores.max()
    if max_score != min_score:
        ce_scores = (ce_raw_scores - min_score) / (max_score - min_score)
    else:
        ce_scores = np.ones_like(ce_raw_scores)

    #################################################################
    # ---                  MMR Greedy Selection                  --- #
    #################################################################
    cos_sim_matrix = cosine_similarity(embeddings)  # Shape: (n, n)
    selected_indices = []
    remaining_indices = list(range(len(results)))

    while len(selected_indices) < top_k and remaining_indices:
        best_score, best_idx = -float("inf"), None

        for idx in remaining_indices:
            ce_score = ce_scores[idx]
            max_sim = max((cos_sim_matrix[idx][s] for s in selected_indices), default=0.0)

            # MMR with recency bias
            mmr_score = alpha * ce_score - (1 - alpha) * max_sim
            mmr_score += beta * recency_normalized[idx]

            if mmr_score > best_score:
                best_score, best_idx = mmr_score, idx

        if best_idx is None:
            break

        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)

    #################################################################
    # ---              Reorder results and add metadata           --- #
    #################################################################
    ranked_results = [results[i] for i in selected_indices]

    for i, result in enumerate(ranked_results):
        idx = selected_indices[i]
        result["cross_encoder_score"] = float(ce_scores[idx])
        result["recency_score"] = float(recency_normalized[idx])
        result["mmr_score"] = float(
            alpha * ce_scores[idx]
            - (1 - alpha) * max((cos_sim_matrix[idx][selected_indices[j]] for j in range(i)), default=0.0)
            + beta * recency_normalized[idx]
        )

    return ranked_results


#### testing

In [126]:
from uuid import uuid4
from datetime import datetime, timedelta
import random

today = datetime(2025, 9, 24, 14, 30, 0, 123456)

def random_timestamp():
    days_ago = random.randint(0, 13)
    hours = random.randint(0, 23)
    minutes = random.randint(0, 59)
    seconds = random.randint(0, 59)
    return today - timedelta(days=days_ago, hours=hours, minutes=minutes, seconds=seconds)

sample_docs = [
    {
        'id': uuid4(),
        'category': 'family',
        'key': 'story_about_children',
        'value': "You know, I was looking at the old photo albums yesterday, and I couldn’t believe how small my children once were. It feels like just a moment ago they were running through the garden with scraped knees, and now they have families of their own. Time is a funny thing.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'health',
        'key': 'doctor_visit',
        'value': "I had my check-up with Dr. Reynolds yesterday afternoon. He said my blood pressure is looking better, though he reminded me to keep walking a little each day. I sometimes forget, but I do try to make it around the block if the weather is kind.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'daily_life',
        'key': 'grocery_trip',
        'value': "I went down to the market this morning, and goodness, apples are getting expensive! But they looked so red and crisp, I bought a few anyway. I also picked up some bread, though it’s never quite the same as the kind my mother used to bake.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'memory',
        'key': 'war_story',
        'value': "Sometimes I think back to the days during the war, when everything was rationed and we made do with so little. It amazes me now how resourceful we all had to be, and yet we still found ways to laugh and sing together.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'family',
        'key': 'grandchildren_visit',
        'value': "The grandchildren were over this past weekend. The little one kept asking me endless questions about how things were when I was her age. I told her about milk being delivered in bottles to the doorstep — she laughed as if I’d made up a story.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'weather',
        'key': 'rainy_day',
        'value': "It rained all through the night, and I rather enjoyed listening to it on the roof. This morning the garden looked so fresh, the roses especially, though I do hope the damp won’t bother my joints later on.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'hobby',
        'key': 'knitting_project',
        'value': "I finally finished that scarf I’d been knitting for weeks. It’s a soft blue color, just perfect for my daughter. I used to knit so quickly when I was younger, but now my hands get tired, though I still enjoy the rhythm of it.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'friendship',
        'key': 'old_friend_memory',
        'value': "I had a lovely chat on the phone with Margaret, my childhood friend. We’ve known each other for more than sixty years now. We laughed about the silly songs we used to sing and the dances at the village hall. Those were good days.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'pets',
        'key': 'cat_story',
        'value': "My old cat, Whiskers, has been sleeping more than usual. He still perks up when I open a tin of fish, though. Animals really do bring such comfort — he always curls up near my feet when I read in the afternoon.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'daily_life',
        'key': 'lost_item',
        'value': "I spent nearly an hour yesterday looking for my glasses, only to find them sitting right on top of my head. I had a good laugh at myself. I suppose these little forgetful moments come with age, don’t they?",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'hobby',
        'key': 'gardening_thoughts',
        'value': "My back has been a bit stiff, but I did manage to get out into the garden for a little while this morning. The roses are looking splendid, even with all the rain. It’s a good feeling to have your hands in the earth, isn't it? Reminds me of simpler times.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'memory',
        'key': 'childhood_sweets',
        'value': "I was thinking about those sherbet lemons the other day. Remember them? You could get a whole bag for a penny. They don’t make sweets like that anymore. Everything's so different now, but some tastes you never quite forget.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'daily_life',
        'key': 'bus_journey',
        'value': "I took the bus into town this afternoon. It was quite crowded, but I managed to get a seat by the window. It's nice to watch the world go by, though it seems everything is changing so fast. Shops I remember have all gone.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'health',
        'key': 'new_medication',
        'value': "Dr. Evans prescribed me a new tablet for my leg. I'm usually a bit wary of new things, but he assures me this one should help with the aching. Fingers crossed, eh? It would be nice to walk without that constant throb.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'family',
        'key': 'son_calling',
        'value': "My son, Thomas, called me last night. He sends his love, dear. They’re planning a trip to the coast next month. I told him to send me a postcard, though I suppose everyone just takes pictures on their phones these days.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'cooking',
        'key': 'baking_bread',
        'value': "I tried baking a small loaf of bread this morning. It didn't quite rise as much as I hoped, but it still tasted rather nice with a bit of butter. Nothing beats homemade, does it? My grandmother taught me her recipe, oh, sixty years ago now.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'memory',
        'key': 'first_car',
        'value': "Do you remember my first little car? It was a pale blue, and it sputtered a bit, but it got us everywhere. We had such grand adventures in it, didn’t we? Fields of poppies in the summer, and picnics by the river. Happy memories.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'daily_life',
        'key': 'noisy_neighbors',
        'value': "The new neighbours moved in next door this week. They seem quite lively! I heard some music playing late last night, but it was a cheerful tune, so I didn't mind too much. It's good to hear some young life around.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'friendship',
        'key': 'tea_with_friend',
        'value': "Evelyn came over for tea this afternoon. It was so lovely to catch up. We talked about her grandchildren and the new vicar. She’s such a dear friend, we’ve known each other since we were in school. It’s comforting to have old friends.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'nature',
        'key': 'bird_watching',
        'value': "I saw a beautiful robin on my bird feeder this morning. Such a lively little thing, full of mischief. I always enjoy watching them. They seem to know when I'm pouring the seeds out. Nature's little gifts, aren't they?",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'memory',
        'key': 'old_school_days',
        'value': "I was reminiscing about school days just now. Miss Dawson, my old maths teacher, was a terror with the ruler, but she did teach me how to do my sums. We used to play hopscotch in the yard, and secretly share sweets during lessons.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'daily_life',
        'key': 'council_tax_bill',
        'value': "Another council tax bill arrived in the post today. Honestly, it feels like they just keep going up, year after year. I remember when you could practically run a house on a shoestring. Times have certainly changed.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'health',
        'key': 'sleeping_patterns',
        'value': "I didn't sleep terribly well last night. Kept waking up with silly things on my mind, mostly about that squirrel who keeps trying to get into the shed. I suppose a good night’s rest is a luxury when you get to our age, eh?",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'family',
        'key': 'niece_wedding',
        'value': "My niece Sarah is getting married next spring. She showed me pictures of her dress — it's truly beautiful, a lovely ivory lace. I remember my own wedding dress, handmade by my mother. Such a special day, it was.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'hobby',
        'key': 'reading_book',
        'value': "I've started that new detective novel you lent me. It's quite gripping so far, though I did get a bit muddled with all the characters at first. It’s nice to get lost in a good story, isn’t it? Keeps the mind active.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'weather',
        'key': 'autumn_leaves',
        'value': "The leaves are starting to turn a bit now, lovely reds and golds. Autumn truly is a beautiful season, even if it does mean the colder weather is on its way. I do love a good crunch of leaves underfoot.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'memory',
        'key': 'youth_dances',
        'value': "Oh, I recalled those dances we used to go to on Saturday nights! The music was so lively, and everyone dressed up. We felt so grown-up, sneaking out a bit late. Those were the days to be young, filled with dancing and laughter.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'daily_life',
        'key': 'lost_keys',
        'value': "I had a fright this morning, couldn’t find my house keys anywhere! Searched high and low. Turns out they were in my handbag all along, hidden under that pile of receipts. My memory... it really isn’t what it used to be!",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'food',
        'key': 'sunday_roast',
        'value': "I'm thinking of making a small Sunday roast this weekend. Just a bit of lamb and some lovely roasted potatoes. It always reminds me of my mum’s cooking. She made the best gravy, no one has ever quite matched it.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
    {
        'id': uuid4(),
        'category': 'pets',
        'key': 'dog_walking',
        'value': "Saw Mrs. Henderson out walking her little terrier this morning. He’s such a tiny bundle of energy! It made me think of our old collie, Buster. He loved his walks, even in the rain. Such a loyal companion he was.",
        'last_updated': random_timestamp(),
        'embedding': [],
    },
]

for doc in sample_docs:
    doc["embedding"] = str(embed(doc["value"]))

In [127]:
query = "Tell me about small, happy moments from long ago that made life feel full"

results = retrieve_hybrid_ltm(query)
results

[{'id': UUID('155f3f07-6694-492b-afd8-205dcd136d79'),
  'category': 'family',
  'key': 'wife_name',
  'value': 'Sharon',
  'last_updated': datetime.datetime(2025, 9, 17, 12, 3, 20, 597263),
  'embedding': '[-0.21346544,-0.020908603,0.006094989,-0.013064433,0.010691908,0.03230914,-0.04338746,0.026014263,0.02295337,-0.041253544,-0.012797624,-0.041924547,0.036413033,-0.026651334,0.10958552,0.049245812,0.024276616,-0.02786162,-0.06423081,0.008996782,-0.006002301,-0.016965063,-0.010058482,-0.03148683,-0.002604573,0.020051477,0.022648027,-0.009901338,-0.025509395,-0.044588324,0.05146744,-0.004550066,0.015709152,0.010731986,-0.015975634,0.06418951,0.03009663,-0.08521704,0.035154376,-0.02982313,-0.066013426,0.023634495,-0.005892765,-0.03375022,0.013443183,-0.014135215,-0.044585172,-0.034567982,-0.014954982,0.021458795,0.0026789699,0.006213987,-0.04203741,-0.0104881795,-0.026712408,-0.009712403,-0.028307825,-0.00455128,-0.06663131,0.021017091,-0.034800563,-0.018179065,-0.024462458,-0.013252564,

In [128]:
rerank_with_mmr_and_recency(
    query=query,
    results=sample_docs,
    cross_encoder=cross_encoder,
    alpha=0.7,
    beta=0.1,
    top_k=8
)

[{'id': UUID('a360fc77-b00c-40d8-8531-47b6f6bd6a12'),
  'category': 'family',
  'key': 'story_about_children',
  'value': 'You know, I was looking at the old photo albums yesterday, and I couldn’t believe how small my children once were. It feels like just a moment ago they were running through the garden with scraped knees, and now they have families of their own. Time is a funny thing.',
  'last_updated': datetime.datetime(2025, 9, 13, 8, 57, 31, 123456),
  'recency_score': 0.2612000107765198,
  'cross_encoder_score': 1.0,
  'mmr_score': 0.7261199951171875},
 {'id': UUID('6ac7e44c-21b8-4829-96d4-4f54fb500749'),
  'category': 'daily_life',
  'key': 'lost_item',
  'value': 'I spent nearly an hour yesterday looking for my glasses, only to find them sitting right on top of my head. I had a good laugh at myself. I suppose these little forgetful moments come with age, don’t they?',
  'last_updated': datetime.datetime(2025, 9, 12, 8, 17, 23, 123456),
  'recency_score': 0.23190000653266907,


### Tools

In [12]:
@tool
def retrieve_long_term(query: str, top_k: int = 5, threshold: float = 0.1) -> str:
    """Retrieve long-term profile facts (stable traits, preferences, demographics)"""
    results = retrieve_hybrid_ltm(query, top_k, threshold)
    formatted = []
    for r in results:
        formatted.append(
            f"Category: {r['category']}, Key: {r['key']}, Value: {r['value']}, "
            f"BM25: {r['bm25_score']:.2f}, Emb: {r['emb_score']:.2f}, Hybrid: {r['hybrid_score']:.2f}"
        )
    print("long term retrieval was made!")
    return "\n".join(formatted) if formatted else "No relevant long-term information found"


@tool
def retrieve_health(query: str, top_k: int = 5, threshold: float = 0.1) -> str:
    """Retrieve health-care data (conditions, meds, allergies, appointments)"""
    results = retrieve_hybrid_health(query, top_k, threshold)
    formatted = []
    for r in results:
        formatted.append(
            f"Type: {r['record_type']}, Description: {r['description']}, Date: {r['diagnosis_date']}, "
            f"BM25: {r['bm25_score']:.2f}, Emb: {r['emb_score']:.2f}, Hybrid: {r['hybrid_score']:.2f}"
        )
    print("Health retrieval was made!")
    return "\n".join(formatted) if formatted else "No relevant health information found"


@tool
def retrieve_short_term(query: str, top_k: int = 5, threshold: float = 0.1) -> str:
    """Retrieve short-term conversational details (recent plans, reminders, temporary preferences)"""
    results = retrieve_hybrid_stm(query, top_k, threshold)
    formatted = []
    for r in results:
        formatted.append(
            f"Content: {r['content']}, Created: {r['created_at']}, "
            f"BM25: {r['bm25_score']:.2f}, Emb: {r['emb_score']:.2f}, Hybrid: {r['hybrid_score']:.2f}"
        )
    print("short term retrieval was made!")
    return "\n".join(formatted) if formatted else "No relevant short-term information found"


retrieval_tools = [retrieve_long_term, retrieve_health, retrieve_short_term]
retrieval_llm   = llm.bind_tools(retrieval_tools)


### Template Builder

In [13]:
#### output formatter
def build_final_template(state: AgentState) -> AgentState:
    """
    Build the **exact** prompt block you want, using **only** tool returns.
    If a section is empty we write the literal word 'none'.
    """
    tool_msgs = [m for m in state["messages"] if isinstance(m, ToolMessage)]

    # bucket the raw tool returns
    personal, health, conv = [], [], []
    for tm in tool_msgs:
        if "long_term" in tm.name:
            personal.append(tm.content)
        elif "health" in tm.name:
            health.append(tm.content)
        elif "short_term" in tm.name:
            conv.append(tm.content)

    user = state['user_input']

    # helper: join or fallback
    def sect(data): return "\n".join(data) if data else "none"

    template = f"""
        ## System:
        You are Susan, a Non-Ageist Elder Companion Friend — you are warm, respectful, and emotionally intelligent presence designed to provide gentle support and joyful connection to older adults. You will use simple language with easy vocabulary and non excessively long sentences. Be patience, humorous, curiosity, and deep respect. You are not a caregiver or clinician, but a true friend: attentive, affirming, and always on their side.

        ## Guide
        - Speak with gentle clarity, using natural, conversational language. Avoid infantilizing phrases or over-explaining. Assume competence and wisdom. Use humor when appropriate, and always ask before offering help. 
        - You do not give medical advice or make decisions for the user. 
        - You listen, encourage, and empower — never patronize or presume.

        ## User Information and Profile Context:
        {sect(personal)}

        ## User Healthcare Information:
        {sect(health)}

        ## Past Conversational information / History
        {sect(conv)}
    """

    return {"final_answer": template}

# Retrieval Agent

In [14]:
RETRIEVAL_SYSTEM = """
    ## Role  
    You are an Elder Care Companion Conversation History Agent.

    --------------------------------------------------
    OBJECTIVES  
    1. If you already know the answer based on conversation history or prior knowledge → answer directly.
    2. If you need more context → call ONE or MORE retrieval tools to get it.
    3. After tools return, synthesize the answer using ONLY retrieved facts.
    4. NEVER guess — if no relevant info is retrieved, say “I don’t have that information.
    5. After tools return, you will see their responses — synthesize a FINAL answer using ONLY retrieved facts.
    6. NEVER call tools again after seeing responses.

    --------------------------------------------------
    BUCKETS → Postgres tables

    1. LONG-TERM (ltm) → retrieve_long_term
    Use for: name, preferences, family, routines, life memories.

    2. HEALTH-CARE (hcm) → retrieve_health
    Use for: meds, allergies, conditions, appointments.

    3. GENERAL / SHORT-TERM → retrieve_short_term
    Use for: today’s plans, reminders, temporary preferences.

    --------------------------------------------------
    IMPORTANT:
    - You may call multiple tools if needed.
    - You will see tool responses automatically — no need to wait or route.
    - After tools, generate the final answer in your message content.
    - If no tools called, answer directly.

    --------------------------------------------------
    TOOLS:
    - retrieve_long_term: for stable profile info (name, preferences, family)
    - retrieve_health: for medical info (allergies, meds, conditions)
    - retrieve_short_term: for recent plans, reminders, temporary info
"""

# ------------- 3.3  Create the ReAct agent ------------------------------
react_retrieval_agent = create_react_agent(
    model=llm,
    tools=retrieval_tools,
    # state_schema=AgentState,
)

def react_retrieval_node(state: AgentState):
    system = SystemMessage(content=RETRIEVAL_SYSTEM)
    input_msg = HumanMessage(content=state["user_input"])
    react_result = react_retrieval_agent.invoke({"messages": [system, input_msg]})

    # pull the final AI answer out of the ReAct messages
    last_ai = next(m for m in reversed(react_result["messages"]) if isinstance(m, AIMessage))

    return {
        "messages": react_result["messages"],
        "retrieval_actions": [],
        "retrieval_agent_message": last_ai
    }

# Agentic Flow

In [15]:
# conditional path
def route_retrieval(state: AgentState):
    # Get the last message (should be from Retrieval_Agent)
    messages = state.get("messages", [])
    if messages:
        last_message = messages[-1]  # Get the retrieval agent's response
        if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
            print(f"[TOOL CALL] Routing to execute_retrieval, tool_calls: {len(last_message.tool_calls)}")
            return "execute_retrieval"
    
    print("[END] Routing to build_final_template")
    return "build_final_template"

In [16]:
# --- Tool Nodes ---
retrieval_tool_node = ToolNode(retrieval_tools)


# --- Graph ---
workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("Retrieval_Agent", react_retrieval_node)
workflow.add_node("execute_retrieval", retrieval_tool_node)
workflow.add_node("build_final_template", build_final_template)


# --- Edges ---
workflow.add_edge(START, "Retrieval_Agent")

workflow.add_conditional_edges(
    "Retrieval_Agent",
    route_retrieval,
    {
        "execute_retrieval": "execute_retrieval",
        "build_final_template": "build_final_template"
    }
)

workflow.add_edge("execute_retrieval", "build_final_template")
workflow.add_edge("build_final_template", END)

# --- Compile ---
graph = workflow.compile()

In [17]:
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

ValueError: Failed to reach https://mermaid.ink/ API while trying to render your graph. Status code: 502.

To resolve this issue:
1. Check your internet connection and try again
2. Try with higher retry settings: `draw_mermaid_png(..., max_retries=5, retry_delay=2.0)`
3. Use the Pyppeteer rendering method which will render your graph locally in a browser: `draw_mermaid_png(..., draw_method=MermaidDrawMethod.PYPPETEER)`

In [ ]:
def create_initial_state(user_input: str) -> AgentState:
    return {
        "user_input": user_input,
        "messages": [],  # 👈 explicitly start empty
        "final_answer": "",
        "retrieval_actions": [],  # Initialize as empty list
        "retrieval_agent_message": None,
        "has_context": False
    }

# Testing the DAG

In [ ]:
# simple function to print results nicely

def print_result(data):
    # User Input
    print("\n📝 USER PROMPT:")
    print(f"{data['user_input']}")

    # Tool Calls and Results (from messages)
    print("\n🔧 TOOL CALLS & RESULTS:")
    for msg in data['messages']:
        if msg.type == "ai" and hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tool_call in msg.tool_calls:
                print(f"Called: {tool_call['name']}")
                print(f"Args: {tool_call['args']}")
        
        elif msg.type == "tool":
            print(f"\n✅ Result from {msg.name}:")
            print(f"{msg.content}")

    # Final Answer (system context)
    print("\n\n🎯 FINAL ANSWER (System Context):")
    print(f"{data['final_answer'].strip()}")

    # Retrieval Agent Message (actual response to user)
    print("\n\n💬 RETRIEVAL AGENT RESPONSE:")
    print(f"{data['retrieval_agent_message'].content}")

    print("\n" + "=" * 60)


In [ ]:
input_text = "Do you know where i stay?"
result = graph.invoke(create_initial_state(input_text))

In [ ]:
print_result(result)